<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/dataframes_lst_per_pixel_allcities_2019_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ee.Authenticate()
ee.Initialize(project="ee-sarahheinz0405")

In [ ]:
!pip install pysal

In [ ]:
# Importieren Sie notwendige Bibliotheken
import ee
import geemap
import folium
from datetime import datetime
import geopandas as gpd
from shapely.geometry import mapping, Point
import pandas as pd
import matplotlib.pyplot as plt
import pysal.lib as ps
import pysal.explore as pe
from esda.getisord import G_Local
from libpysal.weights import KNN
from sklearn.preprocessing import StandardScaler
import numpy as np

# LST-Dataframes erstellen

In [ ]:
def maskClouds(image):
    # QA_PIXEL-Band auswählen
    qa_band = image.select('QA_PIXEL')

    # Bitmasken für Wolkenschatten (Bit 3) und Wolken (Bit 5) definieren
    cloud_shadow_bitmask = (1 << 3)  # 00001000 = 8
    cloud_bitmask = (1 << 5)         # 00100000 = 32

    # Prüfen, ob die relevanten Bits nicht gesetzt sind (d.h. wolkenfreie Pixel)
    mask = qa_band.bitwiseAnd(cloud_shadow_bitmask).eq(0).And(
           qa_band.bitwiseAnd(cloud_bitmask).eq(0))

    # Maske auf das Bild anwenden
    return image.updateMask(mask)


In [ ]:
# Funktion zur Berechnung der LST
def calculateLST(image):
    kelvin = image.select('ST_B10').multiply(0.00341802).add(149.0)  # Radiance -> Kelvin
    lst_celsius = kelvin.subtract(273.15).rename('LST_Celsius')  # Kelvin -> Celsius
    return image.addBands(lst_celsius)


dataframes erstellen, die für jede Stadt die Pixelkoordinaten mitsamt deren LST-Wert enthalten
- für jedes der 5 Jahre einzeln
- für den 5-Jahres-Durchschnitt

zunächst: Beispielhaftes Vorgehen mit Nürnberg

In [ ]:
# Laden Sie die Geometrien der Städte (falls noch nicht geschehen)
staedte_ueber_50tsd_polygone = gpd.read_file("/content/drive/MyDrive/Cold Spots Bayern/grenzen_ueber_50tsd.gpkg")

# Filtern Sie das Polygon für Nürnberg
# IMPORTANT: Double check the exact spelling in your geodataframe
nuernberg_polygon = staedte_ueber_50tsd_polygone[staedte_ueber_50tsd_polygone['name'] == 'Nuremberg'] # Using 'Nuremberg' as per your correction

# Überprüfen Sie, ob das Polygon gefunden wurde
if nuernberg_polygon.empty:
    print("Polygon für Nürnberg nicht gefunden. Bitte überprüfen Sie den Stadtnamen.")
    # If the polygon is not found, the rest of the code for Nuremberg won't run.
    # You might want to add an exit() or sys.exit() here if this is a critical error.
else:
    nuernberg_geometry_shapely = nuernberg_polygon.iloc[0]['geometry']
    nuernberg_region_ee = ee.Geometry(mapping(nuernberg_geometry_shapely))

    # Definieren Sie die Jahre und den Zeitraum (Juni bis August)
    years = range(2019, 2025) # Jahre 2019 bis einschließlich 2024
    cloud_cover_threshold = 10

    all_pixel_data_across_years = []

    # Initialize a geemap map for visualization (if not already initialized)
    try:
        Map # Check if Map object exists from previous cells
    except NameError:
        print("geemap.Map object not found, initializing a new one for visualization.")
        Map = geemap.Map(center=[nuernberg_geometry_shapely.centroid.y, nuernberg_geometry_shapely.centroid.x], zoom=10)


    # Schleife über jedes Jahr
    for year in years:
        start_date = f'{year}-06-01'
        end_date = f'{year}-08-31'

        print(f"\nVerarbeitung von Landsat-Daten für Nürnberg im Zeitraum {start_date} bis {end_date}...")

        # Filtern, Maskieren und LST berechnen
        # Ensure maskClouds is applied to the collection
        landsat_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
            .filterDate(start_date, end_date) \
            .filterBounds(nuernberg_region_ee) \
            .filter(ee.Filter.lt('CLOUD_COVER', cloud_cover_threshold)) \
            .map(maskClouds) \
            .map(calculateLST) # Verwenden Sie Ihre vorhandene calculateLST Funktion

        num_scenes = landsat_collection.size().getInfo()
        print(f"Anzahl der gefundenen Szenen: {num_scenes}")

        # --- Add Visualization Step (for diagnosis) ---
        # You can choose a specific year/scene to visualize here
        # Example: Visualize the first scene found for 2019
        if num_scenes > 0 and year == 2019: # Set the year you want to inspect
             print(f"  Attempting to visualize a masked scene for {year}...")
             # Get the first image from the collection for visualization
             first_image = ee.Image(landsat_collection.first())
             if first_image:
                 try:
                     # Define visualization parameters for LST (adjust min/max as needed)
                     lst_vis_params = {
                         'min': 15,
                         'max': 35,
                         'palette': ['blue', 'purple', 'cyan', 'green', 'yellow', 'red']
                     }
                     # Add the masked LST band to the map, clipped to the city boundary
                     # Use the masked LST band directly from the processed collection
                     Map.addLayer(first_image.select('LST_Celsius').clip(nuernberg_region_ee), lst_vis_params, f'Masked LST {year} (Scene 1)')
                     # Add the city boundary
                     Map.addLayer(nuernberg_region_ee, {'color': 'FF0000'}, 'Nürnberg Boundary')
                     print(f"  Added masked LST layer for {year} to the map. Inspect the map.")
                 except Exception as viz_e:
                     print(f"  Could not add visualization layer for {year}: {viz_e}")
             else:
                 print(f"  Could not get the first image for visualization in year {year}.")
        # --- End Visualization Step ---


        # Wenn Szenen gefunden wurden, verarbeiten Sie sie
        if num_scenes > 0:
            scenes_list = landsat_collection.toList(num_scenes)

            # Function to extract pixel values and coordinates for a single image
            def extract_pixels_from_image(image, region, current_year):
                # Get scene date for logging
                scene_date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
                print(f"  Attempting to extract pixels for scene date: {scene_date}")

                # Select the LST band which is already masked
                masked_lst = image.select('LST_Celsius')

                # Get the mask from the LST band
                mask = masked_lst.mask()

                # Add Latitude and Longitude bands
                lonLat_image = ee.Image.pixelLonLat()

                # Apply the LST mask to the Latitude and Longitude bands
                masked_lonLat = lonLat_image.updateMask(mask)

                # Combine the masked LST and masked LonLat bands
                array_image = masked_lst.addBands(masked_lonLat)


                pixel_data = array_image.reduceRegion(
                    reducer=ee.Reducer.toList(),
                    geometry=region,
                    scale=30,
                    maxPixels=1e9
                )

                # Use getInfo() here to trigger the Earth Engine computation
                # Use .get('key', default_value) to safely access elements
                pixel_data_info = pixel_data.getInfo()

                # Now all lists should ideally have the same length because the mask was applied consistently
                lst_values = pixel_data_info.get('LST_Celsius', []) # Default to empty list
                lats = pixel_data_info.get('latitude', []) # Default to empty list
                lons = pixel_data_info.get('longitude', []) # Default to empty list


                # Add diagnostic prints
                # print(f"    reduceRegion result for {scene_date}: {pixel_data_info}") # Can be very large, use with caution
                print(f"    Length of LST values for scene {scene_date}: {len(lst_values)}")
                print(f"    Length of Lat values for scene {scene_date}: {len(lats)}")
                print(f"    Length of Lon values for scene {scene_date}: {len(lons)}")


                # Check if lists are empty or inconsistent length - this check should now pass if the mask fix worked
                if not lst_values or not lats or not lons or \
                   len(lst_values) != len(lats) or len(lst_values) != len(lons):
                    print(f"    Warnung: Keine oder inkonsistente Pixeldaten für Jahr {current_year}, Szene {scene_date}. Skipping this scene.")
                    return []

                pixel_list = []
                for j in range(len(lst_values)):
                    pixel_list.append({
                        'LST_Celsius': lst_values[j],
                        'longitude': lons[j],
                        'latitude': lats[j],
                        'year': current_year, # Fügen Sie das Jahr hinzu
                        'scene_date': scene_date # Optional: Szene-Datum hinzufügen
                    })

                print(f"  Successfully extracted {len(pixel_list)} pixels for scene date: {scene_date}")
                return pixel_list

            # Schleife über jede Szene in diesem Jahr
            for i in range(num_scenes):
                image = ee.Image(scenes_list.get(i))
                current_year = year
                current_scene_pixel_data = extract_pixels_from_image(image, nuernberg_region_ee, current_year)
                all_pixel_data_across_years.extend(current_scene_pixel_data)

        else:
            print(f"Keine Szenen für das Jahr {year} gefunden.")

    # If you added the map, display it here (in a separate cell ideally)
    # display(Map)


    # Wenn Daten gesammelt wurden, erstellen Sie den GeoDataFrame mit allen Jahresdaten
    if all_pixel_data_across_years:
        df_nuernberg_pixels_all_years = pd.DataFrame(all_pixel_data_across_years)

        # Erstellen Sie die Geometrie-Spalte
        geometry_all_years = [Point(xy) for xy in zip(df_nuernberg_pixels_all_years['longitude'], df_nuernberg_pixels_all_years['latitude'])]
        gdf_nuernberg_pixels_all_years = gpd.GeoDataFrame(df_nuernberg_pixels_all_years, geometry=geometry_all_years)
        gdf_nuernberg_pixels_all_years.crs = "EPSG:4326"

        print("\nGeoDataFrame für Nürnberg mit Pixel-LST für alle Jahre (2019-2024) erstellt:")
        display(gdf_nuernberg_pixels_all_years.head())

        # Sie können diesen GeoDataFrame speichern, wenn Sie möchten
        output_all_years_geojson_path = "/content/drive/MyDrive/Cold Spots Bayern/nuernberg_pixels_lst_2019_2024.geojson"
        gdf_nuernberg_pixels_all_years.to_file(output_all_years_geojson_path, driver='GeoJSON')
        print(f"\nGeoDataFrame wurde gespeichert unter: {output_all_years_geojson_path}")



Verarbeitung von Landsat-Daten für Nürnberg im Zeitraum 2019-06-01 bis 2019-08-31...
Anzahl der gefundenen Szenen: 5
  Attempting to visualize a masked scene for 2019...
  Added masked LST layer for 2019 to the map. Inspect the map.
  Attempting to extract pixels for scene date: 2019-06-24
    Length of LST values for scene 2019-06-24: 207434
    Length of Lat values for scene 2019-06-24: 207434
    Length of Lon values for scene 2019-06-24: 207434
  Successfully extracted 207434 pixels for scene date: 2019-06-24
  Attempting to extract pixels for scene date: 2019-07-26
    Length of LST values for scene 2019-07-26: 207435
    Length of Lat values for scene 2019-07-26: 207435
    Length of Lon values for scene 2019-07-26: 207435
  Successfully extracted 207435 pixels for scene date: 2019-07-26
  Attempting to extract pixels for scene date: 2019-08-11
    Length of LST values for scene 2019-08-11: 193330
    Length of Lat values for scene 2019-08-11: 193330
    Length of Lon values for

,LST_Celsius,longitude,latitude,year,scene_date,geometry
0,35.249363,10.990591,49.540615,2019,2019-06-24,POINT (10.99059 49.54062)
1,33.953933,10.991005,49.540608,2019,2019-06-24,POINT (10.99101 49.54061)
2,34.124834,10.991420,49.540601,2019,2019-06-24,POINT (10.99142 49.5406)
3,34.459800,10.991834,49.540594,2019,2019-06-24,POINT (10.99183 49.54059)
4,35.386084,10.992249,49.540587,2019,2019-06-24,POINT (10.99225 49.54059)



GeoDataFrame wurde gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/nuernberg_pixels_lst_2019_2024.geojson


dataframe für Nürnberg erstellt: nuernberg_pixels_lst_2019_2024: enthält die LST für jeden Pixel jeder der vorhandenen Szenen im Zeitraum Juni-August der Jahre 2019-2024

auf Basis dessen können Mittelwerte, räumliche Nachbarschaften,... berechnet werden

**das jetzt auch für übrige Städte**



In [ ]:
# Laden Sie die Geometrien der Städte
staedte_ueber_50tsd_polygone = gpd.read_file("/content/drive/MyDrive/Cold Spots Bayern/grenzen_ueber_50tsd.gpkg")

# Definieren Sie die Jahre und den Zeitraum (Juni bis August)
years = range(2019, 2025) # Jahre 2019 bis einschließlich 2024
cloud_cover_threshold = 10

# Liste zum Speichern der GeoDataFrames für jede Stadt
all_cities_pixel_gdfs = []

# Funktion zur Wolkenmaskierung (aus deinem Originalcode)
def maskClouds(image):
    qa_band = image.select('QA_PIXEL')
    cloud_shadow_bitmask = (1 << 3)
    cloud_bitmask = (1 << 5)
    mask = qa_band.bitwiseAnd(cloud_shadow_bitmask).eq(0).And(
           qa_band.bitwiseAnd(cloud_bitmask).eq(0))
    return image.updateMask(mask)

# Funktion zur Berechnung der LST (aus deinem Originalcode)
def calculateLST(image):
    kelvin = image.select('ST_B10').multiply(0.00341802).add(149.0)
    lst_celsius = kelvin.subtract(273.15).rename('LST_Celsius')
    return image.addBands(lst_celsius)

# Funktion zum Extrahieren der Pixeldaten für eine einzelne Szene
def extract_pixels_from_image(image, region, current_year, city_name):
    try:
        scene_date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
        # print(f"  Verarbeite Szene für {city_name} vom {scene_date}")

        masked_lst = image.select('LST_Celsius')
        mask = masked_lst.mask()
        lonLat_image = ee.Image.pixelLonLat()
        masked_lonLat = lonLat_image.updateMask(mask)
        array_image = masked_lst.addBands(masked_lonLat)

        pixel_data = array_image.reduceRegion(
            reducer=ee.Reducer.toList(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        )

        pixel_data_info = pixel_data.getInfo()

        lst_values = pixel_data_info.get('LST_Celsius', [])
        lats = pixel_data_info.get('latitude', [])
        lons = pixel_data_info.get('longitude', [])

        if not lst_values or not lats or not lons or len(lst_values) != len(lats) or len(lst_values) != len(lons):
            print(f"    Warnung: Keine oder inkonsistente Pixeldaten für {city_name}, Jahr {current_year}, Szene {scene_date}. Skipping.")
            return []

        pixel_list = []
        for j in range(len(lst_values)):
            pixel_list.append({
                'city': city_name, # Stadtname hinzufügen
                'LST_Celsius': lst_values[j],
                'longitude': lons[j],
                'latitude': lats[j],
                'year': current_year,
                'scene_date': scene_date
            })
        return pixel_list
    except Exception as e:
        print(f"  Fehler bei der Verarbeitung der Szene für {city_name} vom {scene_date}: {e}")
        return []


# Schleife über jede Stadt im GeoDataFrame
for index, row in staedte_ueber_50tsd_polygone.iterrows():
    city_name = row['name']
    city_geometry_shapely = row['geometry']

    # Überprüfen Sie, ob die Geometrie gültig ist
    if not city_geometry_shapely or city_geometry_shapely.is_empty:
        print(f"⚠️ Warnung: Ungültige oder leere Geometrie für Stadt: {city_name}. Überspringe diese Stadt.")
        continue

    # Konvertieren Sie die Shapely-Geometrie in eine Earth Engine Geometrie
    try:
        city_region_ee = ee.Geometry(mapping(city_geometry_shapely))
    except Exception as e:
        print(f"Fehler beim Erstellen der Earth Engine Geometrie für {city_name}: {e}. Überspringe Stadt.")
        continue

    print(f"\n✨ Verarbeitung von Landsat-Daten für Stadt: {city_name}...")

    city_pixel_data_across_years = []

    # Schleife über jedes Jahr für die aktuelle Stadt
    for year in years:
        start_date = f'{year}-06-01'
        end_date = f'{year}-08-31'

        try:
            # Filtern, Maskieren und LST berechnen
            landsat_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                .filterDate(start_date, end_date) \
                .filterBounds(city_region_ee) \
                .filter(ee.Filter.lt('CLOUD_COVER', cloud_cover_threshold)) \
                .map(maskClouds) \
                .map(calculateLST)

            num_scenes = landsat_collection.size().getInfo()
            print(f"    Anzahl der gefundenen Szenen: {num_scenes}") # Optional: Weniger Logs

            if num_scenes > 0:
                scenes_list = landsat_collection.toList(num_scenes)

                # Extrahieren Sie Pixeldaten für jede Szene im Zeitraum
                for i in range(num_scenes):
                    image = ee.Image(scenes_list.get(i))
                    current_year = year

                    current_scene_pixel_data = extract_pixels_from_image(image, city_region_ee, current_year, city_name)
                    city_pixel_data_across_years.extend(current_scene_pixel_data)
            else:
                print(f"    Keine Szenen für {city_name} im Jahr {year} gefunden.") # Optional: Weniger Logs

        except Exception as e:
            print(f"  Fehler bei der Verarbeitung der Landsat-Collection für {city_name} im Jahr {year}: {e}")

    # Wenn Daten für die aktuelle Stadt gesammelt wurden, erstellen Sie den GeoDataFrame
    if city_pixel_data_across_years:
        df_city_pixels_all_years = pd.DataFrame(city_pixel_data_across_years)

        # Erstellen Sie die Geometrie-Spalte
        geometry_all_years = [Point(xy) for xy in zip(df_city_pixels_all_years['longitude'], df_city_pixels_all_years['latitude'])]
        gdf_city_pixels_all_years = gpd.GeoDataFrame(df_city_pixels_all_years, geometry=geometry_all_years)
        gdf_city_pixels_all_years.crs = "EPSG:4326" # Setzen Sie das CRS

        print(f"\nGeoDataFrame für {city_name} mit Pixel-LST für alle Jahre (2019-2024) erstellt. Enthält {len(gdf_city_pixels_all_years)} Pixel.")
        # display(gdf_city_pixels_all_years.head()) # Optional: Die ersten Zeilen anzeigen

        # Füge den erstellten GeoDataFrame der Liste hinzu
        all_cities_pixel_gdfs.append(gdf_city_pixels_all_years)

        # Optional: Speichern Sie den GeoDataFrame für jede Stadt einzeln
        output_city_geojson_path = f"/content/drive/MyDrive/Cold Spots Bayern/{city_name.replace(' ', '_')}_pixels_lst_2019_2024.geojson"
        try:
             gdf_city_pixels_all_years.to_file(output_city_geojson_path, driver='GeoJSON')
             print(f"  GeoDataFrame für {city_name} wurde gespeichert unter: {output_city_geojson_path}")
        except Exception as e:
             print(f"  Fehler beim Speichern des GeoDataFrames für {city_name}: {e}")

    else:
        print(f"Keine Pixeldaten für Stadt {city_name} gefunden.")


# --- Zusammenführen aller städtischen GeoDataFrames ---
print("\nZusammenführen aller städtischen GeoDataFrames zu einem einzigen...")

if all_cities_pixel_gdfs:
    # Verwenden Sie pd.concat, um alle GeoDataFrames in der Liste zusammenzuführen
    # ignore_index=True setzt den Index im zusammengeführten DataFrame zurück
    try:
        gdf_all_cities_merged = pd.concat(all_cities_pixel_gdfs, ignore_index=True)

        print("\nZusammengeführter GeoDataFrame für alle Städte erstellt:")
        display(gdf_all_cities_merged.head())
        print(f"Gesamtanzahl der Pixel im zusammengeführten GeoDataFrame: {len(gdf_all_cities_merged)}")

        # Optional: Speichern Sie den zusammengeführten GeoDataFrame
        output_all_cities_geojson_path = "/content/drive/MyDrive/Cold Spots Bayern/all_cities_pixels_lst_2019_2024_merged.geojson" # Pfad anpassen
        try:
             gdf_all_cities_merged.to_file(output_all_cities_geojson_path, driver='GeoJSON')
             print(f"\nZusammengeführter GeoDataFrame wurde gespeichert unter: {output_all_cities_geojson_path}")
        except Exception as e:
             print(f"\nFehler beim Speichern des zusammengeführten GeoDataFrames: {e}")

    except Exception as e:
        print(f"Fehler beim Zusammenführen der GeoDataFrames: {e}")

else:
    print("Keine GeoDataFrames für Städte erstellt, nichts zum Zusammenführen.")


✨ Verarbeitung von Landsat-Daten für Stadt: Munich...
    Anzahl der gefundenen Szenen: 6
    Anzahl der gefundenen Szenen: 4
    Anzahl der gefundenen Szenen: 1
    Anzahl der gefundenen Szenen: 6
    Anzahl der gefundenen Szenen: 3
    Anzahl der gefundenen Szenen: 3

GeoDataFrame für Munich mit Pixel-LST für alle Jahre (2019-2024) erstellt. Enthält 6792879 Pixel.
  GeoDataFrame für Munich wurde gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/Munich_pixels_lst_2019_2024.geojson

✨ Verarbeitung von Landsat-Daten für Stadt: Nuremberg...
    Anzahl der gefundenen Szenen: 5
    Anzahl der gefundenen Szenen: 4
    Anzahl der gefundenen Szenen: 2
    Anzahl der gefundenen Szenen: 5
    Anzahl der gefundenen Szenen: 1
    Anzahl der gefundenen Szenen: 2

GeoDataFrame für Nuremberg mit Pixel-LST für alle Jahre (2019-2024) erstellt. Enthält 3792279 Pixel.
  GeoDataFrame für Nuremberg wurde gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/Nuremberg_pixels_lst_2019_2024.

,city,LST_Celsius,longitude,latitude,year,scene_date,geometry
0,Munich,29.117435,11.500859,48.248103,2019,2019-06-24,POINT (11.50086 48.2481)
1,Munich,29.732678,11.501263,48.248094,2019,2019-06-24,POINT (11.50126 48.24809)
2,Munich,27.329810,11.500038,48.247851,2019,2019-06-24,POINT (11.50004 48.24785)
3,Munich,27.989488,11.500442,48.247842,2019,2019-06-24,POINT (11.50044 48.24784)
4,Munich,28.686764,11.500846,48.247833,2019,2019-06-24,POINT (11.50085 48.24783)


Gesamtanzahl der Pixel im zusammengeführten GeoDataFrame: 26329255


läuft jetzt erfolgreich für alle Städte durch (ca. 20 min) und speichert für die Sommermonate 2019-2024 alle LST pro Pixel-Werte in zunächst einem gdf pro Stadt und anschließend einem zusammengeführten gdf für alle Städte

gdf enthält also für jede Szene mit < 10% Wolkenbedeckung die LST pro Pixel pro Stadt

In [ ]:
cities_paths = "/content/drive/MyDrive/Cold Spots Bayern/Aschaffenburg_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Augsburg_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Bamberg_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Bayreuth_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Erlangen_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Fürth_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Ingolstadt_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Kempten_(Allgäu)_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Landshut_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Munich_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Nuremberg_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Passau_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Regensburg_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Rosenheim_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Schweinfurt_pixels_lst_2019_2024.geojson", "/content/drive/MyDrive/Cold Spots Bayern/Würzburg_pixels_lst_2019_2024.geojson"

bisher enthalten geodataframes Daten zur jeder einzelnen Szene, erstmal daraus den Mittelwert über ganzen Sommer berechnen (pro Jahr), dann zusammenführen und Cold Spot Algorithmus anwenden

In [ ]:
# Die Liste der GeoJSON-Dateipfade (aus der cities_paths Zelle)
# cities_paths = [...] # Angenommen, diese Variable ist bereits definiert

# Liste zum Speichern der aggregierten GeoDataFrames für jede Stadt
aggregated_city_gdfs = []

print("Verarbeite jede Stadt einzeln und berechne die durchschnittliche Sommer-LST pro Pixel pro Jahr...")

# Schleife über jeden Dateipfad in der cities_paths Liste
for file_path in cities_paths:
    try:
        # Lade den GeoDataFrame für die aktuelle Stadt
        # Versuche, den Stadtnamen aus dem Dateipfad zu extrahieren
        file_name = file_path.split('/')[-1]
        city_name = file_name.replace("_pixels_lst_2019_2024.geojson", "").replace("_", " ")

        print(f"\n✨ Verarbeite Daten für Stadt: {city_name} aus {file_name}...")
        gdf_city = gpd.read_file(file_path)
        print(f"  Geladen: {len(gdf_city)} Pixel-Szene-Einträge.")

        # Überprüfe, ob der GeoDataFrame geladen wurde und nicht leer ist
        if not gdf_city.empty:
            # Gruppieren Sie nach Jahr, Längengrad und Breitengrad und berechnen Sie den Durchschnitt der LST
            # Behalten Sie 'city' als Teil der Gruppierung bei
            aggregated_lst_city = gdf_city.groupby(
                ['year', 'longitude', 'latitude', 'city'] # Gruppieren auch nach Stadt
            )['LST_Celsius'].mean().reset_index()

            # Benennen Sie die aggregierte Spalte um
            aggregated_lst_city = aggregated_lst_city.rename(
                columns={'LST_Celsius': 'avg_summer_LST_Celsius'}
            )

            # Fügen Sie die Geometrie-Spalte wieder hinzu
            unique_pixels_geometry_city = gdf_city[['longitude', 'latitude', 'geometry']].drop_duplicates(
                subset=['longitude', 'latitude']
            ).set_index(['longitude', 'latitude'])

            aggregated_lst_city = aggregated_lst_city.set_index(
                ['longitude', 'latitude']
            ).join(unique_pixels_geometry_city).reset_index()

            # Konvertieren Sie das Ergebnis in einen GeoDataFrame
            # Stellen Sie sicher, dass das CRS des ursprünglichen GeoDataFrames übernommen wird
            if gdf_city.crs is not None:
                gdf_avg_summer_lst_city = gpd.GeoDataFrame(
                    aggregated_lst_city,
                    geometry='geometry',
                    crs=gdf_city.crs
                )
            else:
                print(f"  ⚠️ Warnung: Ursprüngliches CRS für {city_name} nicht gefunden. Setze Standard-CRS (EPSG:4326).")
                gdf_avg_summer_lst_city = gpd.GeoDataFrame(
                    aggregated_lst_city,
                    geometry='geometry',
                    crs="EPSG:4326"
                )


            print(f"  Erstellt aggregierten GeoDataFrame für {city_name} mit {len(gdf_avg_summer_lst_city)} Pixel-Jahres-Einträgen.")
            # display(gdf_avg_summer_lst_city.head()) # Optional: Die ersten Zeilen anzeigen

            # Fügen Sie den aggregierten GeoDataFrame der Liste hinzu
            aggregated_city_gdfs.append(gdf_avg_summer_lst_city)

            # Optional: Speichern Sie den aggregierten GeoDataFrame für die einzelne Stadt
            # output_city_aggregated_geojson_path = f"/content/drive/MyDrive/Cold Spots Bayern/{city_name.replace(' ', '_')}_avg_summer_lst_per_pixel_year.geojson" # Pfad anpassen
            # try:
                 # gdf_avg_summer_lst_city.to_file(output_city_aggregated_geojson_path, driver='GeoJSON')
                 # print(f"  Aggregierter GeoDataFrame für {city_name} wurde gespeichert unter: {output_city_aggregated_geojson_path}")
            # except Exception as e:
                 # print(f"  Fehler beim Speichern des aggregierten GeoDataFrames für {city_name}: {e}")


        else:
            print(f"  GeoDataFrame für {city_name} war leer. Überspringe Aggregation.")

    except Exception as e:
        print(f"  Fehler bei der Verarbeitung der Datei {file_path}: {e}. Überspringe diese Datei.")


# --- Code zum Zusammenführen der aggregierten Stadt-GeoDataFrames ---
print("\n--- Zusammenführen der aggregierten Stadt-GeoDataFrames ---")

if aggregated_city_gdfs:
    try:
        # Verwenden Sie pd.concat, um alle aggregierten GeoDataFrames zu einem einzigen zusammenzuführen
        # ignore_index=True setzt den Index im zusammengeführten DataFrame zurück
        all_cities_avg_summer_lst_per_pixel_year = pd.concat(aggregated_city_gdfs, ignore_index=True)

        print("\nZusammengeführter GeoDataFrame 'all_cities_avg_summer_lst_per_pixel_year' erstellt:")
        display(all_cities_avg_summer_lst_per_pixel_year.head())
        print(f"Gesamtanzahl der Pixel-Jahres-Einträge im zusammengeführten GeoDataFrame: {len(all_cities_avg_summer_lst_per_pixel_year)}")

        # Optional: Speichern Sie den finalen zusammengeführten GeoDataFrame
        output_final_merged_geojson_path = "/content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson" # Pfad anpassen
        try:
             all_cities_avg_summer_lst_per_pixel_year.to_file(output_final_merged_geojson_path, driver='GeoJSON')
             print(f"\nFinaler zusammengeführter GeoDataFrame wurde gespeichert unter: {output_final_merged_geojson_path}")
        except Exception as e:
             print(f"\nFehler beim Speichern des finalen zusammengeführten GeoDataFrames: {e}")

    except Exception as e:
        print(f"Fehler beim Zusammenführen der aggregierten GeoDataFrames: {e}")

else:
    print("Keine aggregierten Stadt-GeoDataFrames erstellt. Zusammenführung nicht möglich.")


Verarbeite jede Stadt einzeln und berechne die durchschnittliche Sommer-LST pro Pixel pro Jahr...

✨ Verarbeite Daten für Stadt: Aschaffenburg aus Aschaffenburg_pixels_lst_2019_2024.geojson...
  Geladen: 525955 Pixel-Szene-Einträge.
  Erstellt aggregierten GeoDataFrame für Aschaffenburg mit 362052 Pixel-Jahres-Einträgen.

✨ Verarbeite Daten für Stadt: Augsburg aus Augsburg_pixels_lst_2019_2024.geojson...
  Geladen: 3531413 Pixel-Szene-Einträge.
  Erstellt aggregierten GeoDataFrame für Augsburg mit 980443 Pixel-Jahres-Einträgen.

✨ Verarbeite Daten für Stadt: Bamberg aus Bamberg_pixels_lst_2019_2024.geojson...
  Geladen: 895498 Pixel-Szene-Einträge.
  Erstellt aggregierten GeoDataFrame für Bamberg mit 543528 Pixel-Jahres-Einträgen.

✨ Verarbeite Daten für Stadt: Bayreuth aus Bayreuth_pixels_lst_2019_2024.geojson...
  Geladen: 594897 Pixel-Szene-Einträge.
  Erstellt aggregierten GeoDataFrame für Bayreuth mit 521674 Pixel-Jahres-Einträgen.

✨ Verarbeite Daten für Stadt: Erlangen aus Erlan

,longitude,latitude,year,city,avg_summer_LST_Celsius,geometry
0,9.080523,50.007251,2019,Aschaffenburg,32.516656,POINT (9.08052 50.00725)
1,9.080524,50.007521,2019,Aschaffenburg,32.417533,POINT (9.08052 50.00752)
2,9.080524,50.007791,2019,Aschaffenburg,32.320120,POINT (9.08052 50.00779)
3,9.080941,50.006711,2019,Aschaffenburg,32.817441,POINT (9.08094 50.00671)
4,9.080941,50.006981,2019,Aschaffenburg,32.648249,POINT (9.08094 50.00698)


Gesamtanzahl der Pixel-Jahres-Einträge im zusammengeführten GeoDataFrame: 10534516

Finaler zusammengeführter GeoDataFrame wurde gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson


nur zusammengeführter gdf mit allen Städten wurde in drive gespeichert => hieraus jetzt immer Städte extrahieren